# Customer Churn Prediction

This notebook covers the end-to-end pipeline for predicting customer churn using the provided Excel dataset.

In [1]:
import pandas as pd
import numpy as np
import joblib
import json
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data

In [2]:
df = pd.read_excel('Telco_customer_churn.xlsx')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


## 2. Clean and Preprocess Data

In [3]:
# Drop columns that are not useful for prediction or cause leakage
columns_to_drop = ['CustomerID', 'Count', 'Country', 'State', 'City', 'Zip Code', 
                   'Lat Long', 'Latitude', 'Longitude', 'Churn Label', 'Churn Score', 
                   'CLTV', 'Churn Reason']
df = df.drop(columns=[col for col in columns_to_drop if col in df.columns], errors='ignore')

# Handle missing Total Charges (if any are stored as strings with spaces)
if 'Total Charges' in df.columns:
    df['Total Charges'] = pd.to_numeric(df['Total Charges'], errors='coerce')
    df['Total Charges'] = df['Total Charges'].fillna(df['Total Charges'].median())

# Encode categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# The target variable is 'Churn Value'
X = df.drop('Churn Value', axis=1)
y = df['Churn Value']

# Scale numerical features
numerical_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges']
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

# Save preprocessing objects
joblib.dump(label_encoders, 'label_encoders.joblib')
joblib.dump(scaler, 'scaler.joblib')

with open('feature_names.json', 'w') as f:
    json.dump(list(X.columns), f)

print("Preprocessing Complete. Features shape:", X.shape)

Preprocessing Complete. Features shape: (7043, 19)


## 3. Train and Compare Models

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

best_model_name = None
best_f1 = -1
best_model = None

for name, model in models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    
    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions)
    rec = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    
    print(f"--- {name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}\n")
    
    if f1 > best_f1:
        best_f1 = f1
        best_model_name = name
        best_model = model

print(f"Best Model based on F1-score: {best_model_name}")

--- Logistic Regression ---
Accuracy:  0.7999
Precision: 0.6746
Recall:    0.5700
F1-score:  0.6179



--- Random Forest ---
Accuracy:  0.7885
Precision: 0.6667
Recall:    0.5100
F1-score:  0.5779



--- XGBoost ---
Accuracy:  0.7842
Precision: 0.6455
Recall:    0.5325
F1-score:  0.5836

Best Model based on F1-score: Logistic Regression


## 4. Save the Best Model

In [5]:
joblib.dump(best_model, 'best_model.joblib')
print(f"Saved {best_model_name} as 'best_model.joblib'")

Saved Logistic Regression as 'best_model.joblib'
